In [3]:
import pandas as pd
import numpy as np
import os

In [4]:
# ---------------------------------------------------------
# ML GOLD TABLE: ml_listing_features
# Purpose: Feature-rich listing-level table for clustering
# ---------------------------------------------------------

import pandas as pd
from pathlib import Path

# Resolve project root (assumes notebook is in /notebooks)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
silver_dir = project_root / "data" / "silver"
gold_dir = project_root / "data" / "gold"
gold_dir.mkdir(parents=True, exist_ok=True)

# --- Load Silver Tables ---
df_listings = pd.read_parquet(silver_dir / "listings_clean.parquet")
df_calendar = pd.read_parquet(silver_dir / "calendar_clean.parquet")
df_reviews = pd.read_parquet(silver_dir / "reviews_clean.parquet")

# --- Calendar Aggregates (Future-Oriented) ---
calendar_agg = (
    df_calendar
    .groupby("listing_id")
    .agg(
        future_calendar_days=("date", "count"),
        future_available_days=("available", "sum"),
        future_avg_price=("price", "mean")
    )
    .assign(
        future_occupancy_rate=lambda df: 1 - (df["future_available_days"] / df["future_calendar_days"])
    )
    .reset_index()
)

# --- Review Aggregates ---
review_agg = (
    df_reviews
    .groupby("listing_id")
    .agg(
        review_count=("id", "count"),
        first_review_date=("date", "min"),
        last_review_date=("date", "max")
    )
    .assign(
        review_span_days=lambda df: (df["last_review_date"] - df["first_review_date"]).dt.days,
        reviews_per_month=lambda df: df["review_count"] / (df["review_span_days"] / 30).replace(0, pd.NA)
    )
    .drop(columns=["review_span_days"])
    .reset_index()
)

# --- Listing Selection ---
listing_cols = [
    "id", "host_id", "room_type", "neighbourhood_cleansed", "price",
    "host_is_superhost", "host_response_rate", "review_scores_rating",
    "beds", "accommodates", "minimum_nights", "maximum_nights",
    "estimated_revenue_l365d", "estimated_occupancy_l365d",
    "number_of_reviews_ltm", "number_of_reviews"
]
df_listings = df_listings[listing_cols]

# --- Superhost Treatment ---
df_listings["host_is_superhost"] = df_listings["host_is_superhost"].fillna("f")
df_listings["host_is_superhost"] = df_listings["host_is_superhost"].map({"t": 1, "f": 0})

# --- Review Score Treatment ---
df_listings["review_scores_rating"] = df_listings["review_scores_rating"].fillna(0)

# --- Merge All Sources ---
ml_listing_features = (
    df_listings
    .merge(calendar_agg, how="left", left_on="id", right_on="listing_id")
    .merge(review_agg, how="left", on="listing_id")
    .drop(columns=["listing_id"])
)

# --- Review Count & Velocity Treatment ---
ml_listing_features["review_count"] = ml_listing_features["review_count"].fillna(0)
ml_listing_features["reviews_per_month"] = ml_listing_features["reviews_per_month"].fillna(0)

C:\Users\emand\AppData\Local\Temp\ipykernel_28412\860054448.py:80: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ml_listing_features["reviews_per_month"] = ml_listing_features["reviews_per_month"].fillna(0)


In [5]:
# ---------------------------------------------------------
# BED IMPUTATION LOGIC
# Purpose: Improve missing or zero bed values using rules + context
# ---------------------------------------------------------

# Create working column
ml_listing_features["beds_imputed"] = ml_listing_features["beds"].copy()

# Rule 1: If accommodates == 1 → beds = 1
ml_listing_features.loc[ml_listing_features["accommodates"] == 1, "beds_imputed"] = 1

# Rule 2: If accommodates == bedrooms → beds = accommodates
# Note: Only applied if 'bedrooms' exists in df_listings
if "bedrooms" in df_listings.columns:
    mask = ml_listing_features["accommodates"] == df_listings["bedrooms"]
    ml_listing_features.loc[mask, "beds_imputed"] = ml_listing_features.loc[mask, "accommodates"]

# Rule 3: Contextual median by neighbourhood + room_type
group_median = (
    ml_listing_features[ml_listing_features["beds_imputed"].notna() & (ml_listing_features["beds_imputed"] > 0)]
    .groupby(["neighbourhood_cleansed", "room_type"])["beds_imputed"]
    .median()
)

def impute_beds(row):
    if pd.isna(row["beds_imputed"]) or row["beds_imputed"] == 0:
        return group_median.get((row["neighbourhood_cleansed"], row["room_type"]), np.nan)
    return row["beds_imputed"]

ml_listing_features["beds_imputed"] = ml_listing_features.apply(impute_beds, axis=1)

# ---------------------------------------------------------
# REVENUE IMPUTATION LOGIC
# Purpose: Fill missing estimated revenue using future price × occupancy
# ---------------------------------------------------------

# Fill missing revenue using future average price and estimated occupancy
ml_listing_features["estimated_revenue_l365d"] = ml_listing_features["estimated_revenue_l365d"].fillna(
    ml_listing_features["future_avg_price"] * ml_listing_features["estimated_occupancy_l365d"]
)

C:\Users\emand\AppData\Local\Temp\ipykernel_28412\2954692851.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["neighbourhood_cleansed", "room_type"])["beds_imputed"]


In [6]:
# ---------------------------------------------------------
# SAVE GOLD TABLE
# Purpose: Persist ML-ready listing features for clustering
# ---------------------------------------------------------

from pathlib import Path

# Resolve project root (assumes notebook is in /notebooks)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent
gold_dir = project_root / "data" / "gold"
gold_dir.mkdir(parents=True, exist_ok=True)

# Save to Gold
output_path = gold_dir / "ml_listing_features.parquet"
ml_listing_features.to_parquet(output_path, index=False)

print(f"✅ Saved ML-ready listing features: {len(ml_listing_features):,} rows → {output_path.name}")

✅ Saved ML-ready listing features: 94,559 rows → ml_listing_features.parquet
